# Proposta 1 — Regressão Logística A × filtrada pelo CHMv2, exatamente nos mesmos 600 pontos cegos (500 m)

Use somente **Area1 para treinar**, e as **mesmas 200 referências externas por região** de Sagrada Família, Lomba Grande e Cristo Rei para avaliar. As referências são pseudo-rótulos automáticos CHMv2 e não rótulos de campo. Os 600 footprints Quickshift e as 600 predições originais estão congelados no repositório.

Este notebook lê os originais grandes **do Google Drive autenticado**, evitando a cota de download público. Não modifica os três arquivos originais.

In [ ]:
!apt-get update -qq && apt-get install -y -qq p7zip-full >/dev/null
!pip -q install numpy pandas rasterio scikit-image scikit-learn requests

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
originais=Path('/content/drive/MyDrive/Imagens do Mestrado')
files=['SAGRADA_FAMÍLIA_CAXIAS DO SUL.zip','LOMBA_GRANDE_NOVO_HAMBURGO.zip','CRISTO_REI_SÃO_LEOPOLDO.rar']
for name in files:
    p=originais/name
    print('OK' if p.is_file() else 'FALTA',p)
    assert p.is_file(), f'Ajuste o caminho para o arquivo original no seu Drive: {p}'

In [ ]:
import os,sys,subprocess
from pathlib import Path
repo=Path('/content/mestrado-arvores-urbanas')
if not (repo/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--branch','feature/validacao-geografica-externa','https://github.com/BrenoBragaGalvao/mestrado-arvores-urbanas.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
script=repo/'experimentos/validacao_externa_regioes/chmv2_verificacao/avaliar_600_pontos_C_vs_A_500m.py'
out=Path('/content/resultado_f1_600_pontos_500m')
env=os.environ.copy()
env['CHM_ORIGINAIS_DRIVE']=str(originais)
env['CHM_F1_OUTPUT']=str(out)
subprocess.run([sys.executable,str(script)],cwd=str(repo),env=env,check=True)

In [ ]:
import json,pandas as pd
manifest=json.loads((out/'manifesto_comparacao.json').read_text(encoding='utf-8'))
n=manifest['diferencas_baseline_A']
print('Divergências em relação às 600 previsões originais do modelo A:',n)
if n>0:
    print('ATENÇÃO: verificar a reprodução espectral antes de publicar os F1 do modelo filtrado.')
df=pd.read_csv(out/'COMPARACAO_F1_MACRO_500m_600_PONTOS.csv')
tab=df.pivot(index='regiao',columns='modelo',values='f1_macro')
tab['ganho_filtrado_menos_original']=tab['C_retreinado_371']-tab['A_raster_original']
display(tab.round(4))
display(df[['regiao','modelo','n_total','n_validos','n_blocos_500','f1_macro','aa','ag','ga','gg']])

In [ ]:
import shutil
dest=Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/outputs/resultado_final_600_pontos_500m')
dest.mkdir(parents=True,exist_ok=True)
for p in list(out.glob('*.csv'))+list(out.glob('*.json')):
    shutil.copy2(p,dest/p.name)
print('Arquivos e métricas salvos em:',dest)